<a href="https://colab.research.google.com/github/aicaptaincode/AI-POWERED-TEXT-SUMMARIZATION/blob/main/AI_POWERED_TEXT_SUMMARIZATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Text → Summary
PDF → Summary
DOCX → Summary
URL → Summary
Gradio frontend → Upload/paste and summarize

AI Text Summarization
1. Install Libraries
2. Import Libraries
3. Check GPU
4. Load BART Model
5. Text Summarization
6. PDF Extraction
7. DOCX Extraction
8. URL Extraction
9. Gradio Frontend
10. Launch Application

Step 1 — Install packages

In [ ]:
!pip install -q "transformers==4.57.1" torch gradio pypdf python-docx requests beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


Step 2 — Import libraries

In [ ]:
import torch
import gradio as gr

from transformers import pipeline
from pypdf import PdfReader
from docx import Document

import requests
from bs4 import BeautifulSoup

Step 3 — Check GPU

In [ ]:
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
    device = 0
else:
    print("GPU not available. Using CPU.")
    device = -1

PyTorch version: 2.11.0+cu128
GPU available: Tesla T4


Step 4 — Load the summarization model

In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device
)

print("Summarization model loaded successfully!")

Device set to use cuda:0


Summarization model loaded successfully!


Create text summarization function

In [ ]:
def summarize_text(text):
    if not text or not text.strip():
        return "Please enter some text."

    text = text.strip()

    # Split text into chunks
    words = text.split()
    chunks = []

    chunk_size = 500

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    summaries = []

    for chunk in chunks:
        try:
            result = summarizer(
                chunk,
                max_length=130,
                min_length=30,
                do_sample=False
            )

            summaries.append(result[0]["summary_text"])

        except Exception as e:
            summaries.append(f"Error: {str(e)}")

    # If multiple chunks, summarize the combined summaries
    final_summary = " ".join(summaries)

    if len(summaries) > 1:
        try:
            result = summarizer(
                final_summary,
                max_length=180,
                min_length=50,
                do_sample=False
            )

            final_summary = result[0]["summary_text"]

        except Exception:
            pass

    return final_summary

Test text summarization

In [ ]:
text = """
Artificial intelligence is rapidly transforming many industries.
Machine learning allows computers to learn patterns from large amounts
of data without being explicitly programmed for every task. Deep learning,
a subset of machine learning, uses neural networks with many layers to
process complex information such as images, speech, and natural language.

Generative AI has further expanded the capabilities of artificial
intelligence by allowing systems to create text, images, audio, video,
and computer code. Large language models are now being used for customer
support, education, software development, research, and content creation.

However, artificial intelligence also introduces challenges such as
privacy concerns, bias, security risks, misinformation, and the need for
responsible development. Researchers and organizations are working to
develop AI systems that are useful, reliable, transparent, and safe.
"""

summary = summarize_text(text)

print(summary)

Machine learning allows computers to learn patterns from large amounts of data. Deep learning uses neural networks with many layers to process complex information. Generative AI allows systems to create text, images, audio, video, and computer code.


Step 5 — PDF text extraction

In [ ]:
def extract_text_from_pdf(pdf_file):
    try:
        reader = PdfReader(pdf_file)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        return text

    except Exception as e:
        return f"PDF extraction error: {str(e)}"

Summarize PDF

In [ ]:
def summarize_pdf(pdf_file):
    if pdf_file is None:
        return "Please upload a PDF file."

    text = extract_text_from_pdf(pdf_file)

    if not text.strip():
        return "Could not extract text from this PDF."

    return summarize_text(text)

You can test it in Colab after uploading a PDF:

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]

print(summarize_pdf(pdf_filename))

Step 6 — DOCX document extraction

In [ ]:
def extract_text_from_docx(docx_file):
    try:
        document = Document(docx_file)

        text = ""

        for paragraph in document.paragraphs:
            if paragraph.text.strip():
                text += paragraph.text + "\n"

        return text

    except Exception as e:
        return f"DOCX extraction error: {str(e)}"

Summarize DOCX

In [ ]:
def summarize_docx(docx_file):
    if docx_file is None:
        return "Please upload a DOCX file."

    text = extract_text_from_docx(docx_file)

    if not text.strip():
        return "Could not extract text from this document."

    return summarize_text(text)

Step 7 — URL extraction

In [ ]:
def extract_text_from_url(url):

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # Remove unnecessary elements
        for element in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside"
        ]):
            element.decompose()

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        return text

    except Exception as e:
        return f"URL extraction error: {str(e)}"

Summarize URL

In [ ]:
def summarize_url(url):

    if not url or not url.strip():
        return "Please enter a URL."

    text = extract_text_from_url(url)

    if text.startswith("URL extraction error"):
        return text

    if not text.strip():
        return "Could not extract text from this webpage."

    return summarize_text(text)

Test:

In [ ]:
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"

summary = summarize_url(url)

print(summary)

Step 8 — Create the Gradio frontend

In [ ]:
import gradio as gr

Text tab

In [ ]:
text_input = gr.Textbox(
    label="Enter Text",
    placeholder="Paste your text here...",
    lines=15
)

text_output = gr.Textbox(
    label="Summary",
    lines=8
)

text_button = gr.Button("Summarize Text")

Complete Gradio application

In [ ]:
with gr.Blocks(title="AI Text Summarizer") as app:

    gr.Markdown(
        """
        # 🧠 AI Text Summarizer

        Summarize **text, PDF files, Word documents, and webpages**
        using Artificial Intelligence.
        """
    )

    # -------------------------
    # TEXT TAB
    # -------------------------

    with gr.Tab("📝 Text"):

        text_input = gr.Textbox(
            label="Enter Text",
            placeholder="Paste your text here...",
            lines=15
        )

        text_button = gr.Button(
            "Summarize Text"
        )

        text_output = gr.Textbox(
            label="Summary",
            lines=8
        )

        text_button.click(
            fn=summarize_text,
            inputs=text_input,
            outputs=text_output
        )

    # -------------------------
    # PDF TAB
    # -------------------------

    with gr.Tab("📄 PDF"):

        pdf_input = gr.File(
            label="Upload PDF",
            file_types=[".pdf"],
            type="filepath"
        )

        pdf_button = gr.Button(
            "Summarize PDF"
        )

        pdf_output = gr.Textbox(
            label="PDF Summary",
            lines=10
        )

        pdf_button.click(
            fn=summarize_pdf,
            inputs=pdf_input,
            outputs=pdf_output
        )

    # -------------------------
    # DOCX TAB
    # -------------------------

    with gr.Tab("📑 Word Document"):

        docx_input = gr.File(
            label="Upload Word Document",
            file_types=[".docx"],
            type="filepath"
        )

        docx_button = gr.Button(
            "Summarize Document"
        )

        docx_output = gr.Textbox(
            label="Document Summary",
            lines=10
        )

        docx_button.click(
            fn=summarize_docx,
            inputs=docx_input,
            outputs=docx_output
        )

    # -------------------------
    # URL TAB
    # -------------------------

    with gr.Tab("🌐 URL"):

        url_input = gr.Textbox(
            label="Enter Website URL",
            placeholder="https://example.com"
        )

        url_button = gr.Button(
            "Summarize Webpage"
        )

        url_output = gr.Textbox(
            label="Webpage Summary",
            lines=10
        )

        url_button.click(
            fn=summarize_url,
            inputs=url_input,
            outputs=url_output
        )


app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://28fa5facd06a2120a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
